In [1]:
import ibm_db
import ibm_db_dbi
import configparser
import pandas as pd

config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\DB2STAT_connect.ini')

conn_str=config['DB2STAT']['conn_str']
ibm_db_conn = ibm_db.connect(conn_str,'','')
print("SUCCESS")

In [33]:
# Fetch data using ibm_db_dbi
QUERY="""SELECT  
        P.POI_ID,
        coalesce(L.LOC_ID,0) AS CITY_ID,
        P.POI_LAT* 0.000001 AS CITY_CENTER_LATITUDE,
        P.POI_LAT,
        P.POI_LON* 0.000001 AS CITY_CENTER_LONGITUDE,
        P.POI_LON,
        MAX(P.CTS) AS LATEST_TIME,
        HD.DESC_STRING AS CITY_NAME
FROM HRSDB.HTL_POI_ALL_DA P
LEFT JOIN HRSDB.HTL_LOC_POI_ALL_LU L ON L.POI_ID=P.POI_ID
LEFT JOIN HRSDB.HTL_DESCRIPTION_ALL_DA HD ON HD.DESC_ID=P.POI_ID
WHERE POIGR_ID IN (70)
AND HD.DESC_ID_TYPE=5
AND DESC_DEFAULT=1
GROUP BY 
P.POI_ID,
L.LOC_ID,
P.POI_LAT,
P.POI_LON,
HD.DESC_STRING"""

conn = ibm_db_dbi.Connection(ibm_db_conn)
df = pd.read_sql(QUERY, conn)
print(df.head())

In [93]:
import re

# CXL_days
df['CXL_days'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=P)(.*)(?=D)', x)).str[0].astype(float)
df['CXL_days'] = df['CXL_days'].fillna(0)

# CXL_hours
df['CXL_hours'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=T)(.*)(?=H)', x.replace('-', ''))).str[0].astype(float)
df['CXL_hours'] = df['CXL_hours'].fillna(0)

# CXL_minutes
df['CXL_minutes'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=H)(.*)(?=M)', x.replace('-', ''))).str[0].astype(float)
df['CXL_minutes'] = df['CXL_minutes'].fillna(0)

# CXL_seconds
df['CXL_seconds'] = df['CNCLLTN_POLICY_RELATIV_DEADLIN'].apply(lambda x: re.findall('(?<=M)(.*)(?=S)', x.replace('-', ''))).str[0].astype(float)
df['CXL_seconds'] = df['CXL_seconds'].fillna(0)

# Total_hours
df['Total_hours'] = ((df.CXL_days*24) + df.CXL_hours)

# Total_hours_6
df['Total_hours_6'] = (((df.CXL_days*24) + df.CXL_hours) - 6)
df['Total_hours_6'] = df['Total_hours_6'].apply(lambda x: 0 if x <0  else x)

df.head()

In [30]:
df.dtypes

In [ ]:
df['CITY_ID'].apply(int)

In [4]:
df.to_csv('C:\\Users\\USER\\Documents\\db2_city_center_geocodes.csv', 
          sep = '|', 
          header = True,
          encoding='utf-8',
          index=False)

In [29]:
#df.CITY_ID = df.CITY_ID.astype(int)
#df['CITY_ID'].astype(str).astype(int)
df["CITY_ID"]=df["CITY_ID"].astype(int)

In [23]:
df.CITY_ID =df.CITY_ID.fillna(0, inplace=True)

In [31]:
df.head()